In [0]:
%run ../utils

In [0]:
# Explode emails array from leads_processed_contacts into leads_processed_contacts_emails
# Schema discovered dynamically — no hard-coded field names from the struct.
# "email_data.*" expands ALL fields whatever they are.

from pyspark.sql.functions import col, explode_outer, md5, concat_ws
import time

start = time.time()

print(f"{'='*80}")
print(f"📥 leads_processed_contacts → leads_processed_contacts_emails")
print(f"{'='*80}")

# Read from contacts silver table
df_contacts = spark.table("crm_ingestion.silver.leads_processed_contacts")

# Explode the emails array — one row per email per contact
# "email_data.*" dynamically expands ALL struct fields (no hard-coding)
df_emails = (
    df_contacts
    .select(
        col("leads_processed_id"),
        col("id").alias("leads_processed_contacts_id"),
        col("bronze_insert_date"),
        col("bronze_updated_at"),
        explode_outer(col("emails")).alias("email_data")
    )
    .select(
        "leads_processed_id",
        "leads_processed_contacts_id",
        "bronze_insert_date",
        "bronze_updated_at",
        "email_data.*"  # Discover all struct fields dynamically
    )
)

# Add record_hash for deduplication (hash all discovered columns)
hash_cols = [col(c) for c in df_emails.columns]
df_emails = df_emails.withColumn("record_hash", md5(concat_ws("|", *hash_cols)))

# Write to silver table
(
    df_emails.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("crm_ingestion.silver.leads_processed_contacts_emails")
)

# Summary
row_count = df_emails.count()
print(f"\n✅ Table created: crm_ingestion.silver.leads_processed_contacts_emails")
print(f"🔢 Row count: {row_count:,}")
print(f"\n📋 SCHEMA:")
spark.table("crm_ingestion.silver.leads_processed_contacts_emails").printSchema()
print(f"\n📊 SAMPLE ROWS:")
display(spark.table("crm_ingestion.silver.leads_processed_contacts_emails").limit(10))
print(f"\n⏱️ Completed in {time.time() - start:.1f}s")
print(f"{'='*80}")